# Missing Values and Outliers

## Import Libraries and Loading

In [ ]:
%matplotlib inline

%pip install missingno --quiet

import sys, os

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.loader import load_tables
from utils.schema_guard import SchemaContract
from utils.missingness import MissingnessAnalyzer
from utils.outliers import OutlierDetector
import missingno as msno
import matplotlib.ticker as mticker

os.makedirs(MISSING_REPORT_DIR, exist_ok=True)
os.makedirs(OUTLIER_REPORT_DIR, exist_ok=True)

dfs = load_tables("reconciled", normalize_cols="lower")

contract = SchemaContract()
for name, df in dfs.items():
    contract.register(name, df)

print(f"\nRegistered {len(contract.registered_tables)} table schemas.")

pre_report = contract.validate_all(dfs)
if pre_report["failures"] == 0:
    print("Pre-analysis schema validation PASSED - loaded tables match the registered schemas.")
else:
    print("Pre-analysis schema validation FAILED - unexpected structural issues detected!")
    for entry in pre_report["details"]:
        if entry["status"] != "PASS":
            print(f"  {entry}")

## Missingness and Outlier Analysis

In [ ]:
DARK = "#2C3E50"
RED = "#E74C3C"
BLUE = "#3498DB"
FENCE = "#7F8C8D"
LIGHT = "#D6EAF8"

all_summaries = []

for table_name, df in dfs.items():
    config = ANALYTICS_CONFIG.get(table_name, {})

    print(f"\n{'='*60}")
    print(f"  {table_name.upper()}")
    print(f"{'='*60}")

    n_sample = config.get("sample_size")
    df_work = df.sample(n_sample, random_state=42) if n_sample and len(df) > n_sample else df

    # PHASE 1: MISSING VALUES
    analyzer = MissingnessAnalyzer(df_work, table_name)
    summary = analyzer.summary()

    if not summary.empty:
        print("--- Missingness Summary ---")
        print(summary[["missing_count", "missing_pct", "status"]].to_string())

        target = config.get("missing_target")
        features = config.get("missing_features", [])
        if target and target in df_work.columns and features:
            corr_df = analyzer.missing_correlation(target, features)
            if not corr_df.empty:
                print(f"\nMissingness correlation (target: {target}):")
                print(corr_df.to_string(index=False))

        mcar_pair = config.get("mcar_pair")
        if mcar_pair:
            col_t, col_g = mcar_pair
            if col_t in df_work.columns and col_g in df_work.columns:
                r = analyzer.test_mcar_chi2(col_t, col_g)
                print(f"\nMCAR chi-squared ({col_t} vs {col_g}):")
                print(f'  p={r["p_value"]}  {r["significance"]}  ->  {r["verdict"]}')

        if target and target in df_work.columns and features:
            r = analyzer.test_mar_logistic(target, features)
            print(f"\nMAR logistic ({target}):")
            print(f'  pseudo_R2={r.get("pseudo_r2")}  ->  {r.get("verdict")}')

        if target and target in df_work.columns:
            mech = analyzer.classify_mechanism(
                target_col=target,
                group_cols=[mcar_pair[1]] if mcar_pair else [],
                predictor_cols=features if features else [],
            )
            print(f"\n  Missingness mechanism ({target}): " f"{mech.get('mechanism')} " f"(confidence={mech.get('confidence')}) " f"- {mech.get('mcar_test', '')}")

        # Missingness visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1.5, 1]})
        fig.suptitle(f"{table_name} - Missing Values", fontsize=13, fontweight="bold", color=DARK, x=0.02, ha="left")

        vis_sample = df_work.sample(min(3000, len(df_work)), random_state=42)
        msno.matrix(vis_sample, ax=axes[0], sparkline=False, color=(0.18, 0.28, 0.49), fontsize=9)
        axes[0].set_title("Matrix  (white = missing)", fontsize=10, color=DARK, pad=8, loc="left")
        for spine in axes[0].spines.values():
            spine.set_visible(False)

        msno.bar(df_work, ax=axes[1], color=BLUE, fontsize=9)
        axes[1].set_title("Completeness", fontsize=10, color=DARK, pad=8, loc="left")
        axes[1].set_yticks([])
        axes[1].tick_params(axis="x", length=0, labelcolor="#555555")
        for spine in axes[1].spines.values():
            spine.set_visible(False)

        plt.subplots_adjust(wspace=0.15)
        plt.tight_layout()
        out = MISSING_REPORT_DIR / f"Missing_{table_name}.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print(f"Saved: {out}")

    else:
        print("No missing values found.")

    # PHASE 2: OUTLIERS
    uni_cols = config.get("outlier_uni", [])
    multi_cols = config.get("outlier_multi", [])

    if not uni_cols and not multi_cols:
        print("No numeric columns configured for outlier detection.")
        continue

    detector = OutlierDetector(df_work, table_name)

    for col_cfg in uni_cols:
        col, mult = col_cfg if isinstance(col_cfg, tuple) else (col_cfg, 1.5)
        if col in df_work.columns:
            detector.detect_iqr(col, multiplier=mult)
            detector.detect_modified_zscore(col, threshold=3.5)

    available_multi = [c for c in multi_cols if c in df_work.columns]
    if len(available_multi) >= 2:
        detector.detect_isolation_forest(available_multi, contamination=0.02)

    if not detector.outliers.empty:
        consensus = detector.consensus_vote(min_votes=2)
        n_consensus = int(consensus.sum())
        pct_consensus = n_consensus / len(df_work) * 100
        print(f"\n  Consensus outliers (>=2 methods): {n_consensus:,} ({pct_consensus:.2f}%)")

    scorecard = detector.get_summary()
    if not scorecard.empty:
        print("\n--- Outlier Scorecard ---")
        print(scorecard[["method", "column", "n_flagged", "pct_flagged"]].to_string(index=False))
        scorecard.insert(0, "table", table_name)
        all_summaries.append(scorecard)

    df_flagged = df_work.join(detector.outliers)

    # Outlier visualization
    first_uni = uni_cols[0] if uni_cols else None
    first_uni_col = first_uni[0] if isinstance(first_uni, tuple) else first_uni

    has_uni = bool(first_uni_col and first_uni_col in df_work.columns)
    has_multi = "IsoForest_Outlier" in df_flagged.columns and len(available_multi) >= 2

    if not has_uni and not has_multi:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f"{table_name} - Outlier Detection", fontsize=13, fontweight="bold", color=DARK, x=0.02, ha="left")

    if has_uni:
        ax = axes[0]
        dat = df_work[first_uni_col].dropna()
        iqr_row = scorecard[(scorecard["column"] == first_uni_col) & (scorecard["method"].str.startswith("IQR"))]
        lo = iqr_row["lower_fence"].values[0] if len(iqr_row) else None
        hi = iqr_row["upper_fence"].values[0] if len(iqr_row) else None
        ax.boxplot(
            dat,
            vert=False,
            patch_artist=True,
            widths=0.4,
            boxprops=dict(facecolor=LIGHT, color=DARK, linewidth=1, alpha=0.85),
            medianprops=dict(color=RED, linewidth=2),
            whiskerprops=dict(color=DARK, linewidth=0.8),
            capprops=dict(color=DARK, linewidth=0.8),
            flierprops=dict(marker="o", markerfacecolor=RED, markeredgecolor="none", markersize=4, alpha=0.5),
        )
        if hi is not None:
            ax.axvline(hi, color=FENCE, linestyle="--", linewidth=0.9, label=f"Upper fence: {hi:.1f}")
        if lo is not None and lo > dat.min():
            ax.axvline(lo, color=FENCE, linestyle="--", linewidth=0.9, label=f"Lower fence: {lo:.1f}")
        ax.set_title(f"Univariate - {first_uni_col}  (IQR)", fontsize=10, fontweight="bold", color=DARK, pad=10, loc="left")
        ax.set_xlabel(first_uni_col, fontsize=9, color="#555555")
        ax.set_yticks([])
        ax.spines["left"].set_visible(False)
        # Use integer ticks for integer-valued columns (e.g. year_of_manufacture)
        if dat.dropna().apply(lambda x: x == int(x)).all():
            ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
            ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))
        if lo is not None or hi is not None:
            ax.legend(fontsize=8, frameon=False)
    else:
        axes[0].axis("off")

    if has_multi:
        ax = axes[1]
        if {"latitude", "longitude"}.issubset(available_multi):
            x_col = "longitude"
            y_col = "latitude"
        else:
            x_col = available_multi[0]
            y_col = available_multi[1]
        normal = df_flagged[~df_flagged["IsoForest_Outlier"]]
        anoms = df_flagged[df_flagged["IsoForest_Outlier"]]
        ax.scatter(normal[x_col], normal[y_col], color="#BDC3C7", alpha=0.35, s=12, label="Normal", edgecolors="none")
        ax.scatter(anoms[x_col], anoms[y_col], color=RED, alpha=0.8, s=30, label="Anomaly", edgecolors="white", linewidth=0.5, zorder=3)
        ax.set_title(f"Multivariate - {y_col} vs {x_col}  (Isolation Forest)", fontsize=10, fontweight="bold", color=DARK, pad=10, loc="left")
        ax.set_xlabel(x_col, fontsize=9, color="#555555")
        ax.set_ylabel(y_col, fontsize=9, color="#555555")
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.legend(loc="upper left", frameon=False, fontsize=9, labelcolor="#555555")
        ax.spines["bottom"].set_color("#DDDDDD")
        ax.spines["left"].set_color("#DDDDDD")
        ax.grid(True, alpha=0.3, color="#CCCCCC", linewidth=0.5)
        ax.tick_params(axis="both", length=0, labelsize=9, labelcolor="#555555")
    else:
        axes[1].axis("off")

    plt.subplots_adjust(wspace=0.25)
    plt.tight_layout()
    out = OUTLIER_REPORT_DIR / f"Outliers_{table_name}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {out}")

print("\nLoop complete.")

## Null Co-occurrence Analysis (Weather_Observation vs Flight)

In [ ]:
if "Weather_Observations" in dfs and "Flights" in dfs:
    from utils.config import WEATHER_COLS

    wo_analyzer = MissingnessAnalyzer(dfs["Weather_Observations"], "Weather_Observations")
    result = wo_analyzer.null_cooccurrence_flight_weather(
        flight_df=dfs["Flights"],
        weather_cols=[c for c in WEATHER_COLS if c in dfs["Weather_Observations"].columns],
    )
    print("\n--- Null Co-occurrence: Weather_Observations <-> Flights ---")
    for k, v in result.items():
        print(f"  {k}: {v}")

## Global Outlier Report

In [ ]:
if all_summaries:
    report = pd.concat(all_summaries, ignore_index=True)
    report_path = OUTLIER_REPORT_DIR / "outlier_report_all_tables.csv"
    report.to_csv(report_path, index=False)
    print("Global outlier report:")
    print(report[["table", "method", "column", "n_flagged", "pct_flagged"]].to_string(index=False))
    print(f"\nSaved: {report_path}")
else:
    print("No outlier results to report.")

## Post-Analysis Schema Validation

In [ ]:
# Verify that the analysis did not inadvertently modify any table
report = contract.validate_all(dfs)
if report["failures"] == 0:
    print("Schema validation PASSED - no structural changes after analysis.")
else:
    print("Schema validation FAILED - unexpected structural changes detected!")
    for entry in report["details"]:
        if entry["status"] != "PASS":
            print(f"  {entry}")